In [1]:
import torch
import json
from transformers import set_seed
from qwen_vl_utils import process_vision_info
import pandas as pd


from train.qwen_vl_re_see_vision.modeling_qwen_vl import Qwen2_5_VLForConditionalGeneration
from train.qwen_vl_re_see_vision.Qwen2_5_VLProcessor import Qwen2_5_VLProcessor


checkpoint = '/data/dongxinpeng/my_former_work/work/checkpoint-output-self+cross/Qwen2.5-VL-7B-Instruct/ShareGPT4V/my_sharegpt4v_instruct_check_max_3/last_hidden_state_merge-with_null_vision_token-for_cycle-inherit_from_pretrain/train_param:+llm+llm_head+vision_tower+merger+inf_former'

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        checkpoint,
        torch_dtype=torch.bfloat16,
        attn_implementation='flash_attention_2',
        device_map="cuda:1",
    )
processor = Qwen2_5_VLProcessor.from_pretrained(checkpoint)

model.config.my_decoder_mode = 'last_hidden_state_merge-with_null_vision_token-for_cycle-inherit_from_pretrain'

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.50, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [34]:
# image_path = '/data/dongxinpeng/datasets/MME/images/OCR/0002.jpg'
image_path = '/data/dongxinpeng/my_former_work/work/test_images/a.png'
# 0acc12d1261d99ca.jpg.  08c43cd8f2be6709
# question = 'Describe this image in detial.'
question = 'what is unusual about this image?'
# question = 'Who is driving the car?'
# question = 'What happen in the image?'

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image_path,
            },
            {"type": "text", "text": question},
        ],
    }
]
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to(model.device)
generated_ids = model.generate(**inputs, 
                            max_new_tokens=1024,
                            temperature=1.0,  # Increased from 0.7
                            top_k=1,         # Added top_k sampling
                            top_p=0.001,       # Added nucleus sampling
                            do_sample=True ,   # Enable sampling
                            # repetition_penalty=1.05,
                            )
generated_ids_trimmed = [out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)
out_ans = output_text[0]
print(out_ans)

The unusual aspect of this image is that a man is standing on the back of a taxi, ironing a shirt while the car is in motion. This is not a typical scenario, as people usually do not perform tasks like ironing while riding in a moving vehicle. The man's actions are both unexpected and potentially dangerous, as he is balancing on the taxi and handling an iron, which can be a fire hazard if not used properly.
